# Smart Bakery – Keşifsel Veri Analizi (EDA)

Bu defter dbt `marts` katmanındaki tabloları (veri sözlüğü: `docs/DATA_DICTIONARY.md`) inceler:

1. Tablo boyutları ve veri kalitesi
2. Kontrol sayıları (mutabakat): tablolar birbiriyle ve ham veriyle tutarlı mı?
3. Ana göstergeler (KPI)
4. Satış, israf/fire, hava durumu, müşteri (RFM) ve üretim tavsiyesi analizleri

**Çalıştırmadan önce (Colab):** sol paneldeki klasör simgesine tıkla ve repodaki `data/marts/` klasöründeki 7 `.parquet` dosyasını yükle.
Sonra menüden **Çalışma zamanı → Tümünü çalıştır** de.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 1. Veriyi yükle ve tablo boyutlarını kontrol et

In [ ]:
DATA_DIR = "."   # Colab'de dosyaları sol panelden yüklediysen "." doğrudur

BEKLENEN_SATIR = {          # README'deki rakamlar
    "fct_satislar": 19321,
    "fct_israf_ve_fire_analizi": 16563,
    "fct_ideal_uretim_tavsiyesi": 735,
    "dim_musteriler": 500,
    "dim_musteri_rfm": 500,
    "dim_urunler": 35,
    "dim_tarih": 731,
}

d = {t: pd.read_parquet(f"{DATA_DIR}/{t}.parquet") for t in BEKLENEN_SATIR}

rapor = pd.DataFrame({
    "satir": {t: len(df) for t, df in d.items()},
    "sutun": {t: df.shape[1] for t, df in d.items()},
    "beklenen_satir": BEKLENEN_SATIR,
})
rapor["durum"] = np.where(rapor["satir"] == rapor["beklenen_satir"], "OK", "FARK")
rapor

In [ ]:
fct       = d["fct_satislar"].copy()
israf     = d["fct_israf_ve_fire_analizi"].copy()
ideal     = d["fct_ideal_uretim_tavsiyesi"].copy()
musteri   = d["dim_musteriler"].copy()
rfm       = d["dim_musteri_rfm"].copy()
urun      = d["dim_urunler"].copy()
tarih     = d["dim_tarih"].copy()

# tarih sütunlarını aynı tipe getir (birleştirmelerde sorun çıkmasın)
fct["siparis_tarihi"]   = pd.to_datetime(fct["siparis_tarihi"])
israf["siparis_tarihi"] = pd.to_datetime(israf["siparis_tarihi"])
tarih["tarih"]          = pd.to_datetime(tarih["tarih"])

## 2. Veri kalitesi: eksik değerler ve temel istatistikler

In [ ]:
for ad, df in d.items():
    eksik = df.isna().sum()
    eksik = eksik[eksik > 0]
    print(f"{ad:30s} -> " + ("eksik değer yok" if eksik.empty else eksik.to_dict().__str__()))

In [ ]:
fct[["adet", "birim_fiyat", "indirim_orani", "toplam_tutar", "kar", "fire_orani", "musteri_puani"]].describe()

## 3. Kontrol sayıları (mutabakat)

Aynı büyüklüğü iki farklı tablodan hesaplayıp karşılaştırıyoruz. **FARK** çıkan satır otomatik olarak hata demek değildir (yuvarlama olabilir),
ama nedeni açıklanabilmelidir.

Not: `fct_israf_ve_fire_analizi` mağaza + ürün + gün bazında özetlendiği için satır sayısı `fct_satislar`'dan (sipariş kalemi bazında) azdır.

In [ ]:
sonuc = []

def kontrol(ad, a, b, mutlak_tol=0.0):
    tol = max(mutlak_tol, 1e-4 * abs(b))
    sonuc.append({"kontrol": ad, "A": a, "B": b, "fark": a - b, "durum": "OK" if abs(a - b) <= tol else "FARK"})

kontrol("Sipariş kalemi sayısı (A: fct_satislar, B: ham veri)", len(fct), 19321)
kontrol("Tekil sipariş sayısı (A: fct_satislar, B: ham veri)", fct["siparis_id"].nunique(), 6576)
kontrol("Ciro (A: fct_satislar, B: dim_musteriler)", fct["toplam_tutar"].sum(), musteri["toplam_harcama"].sum(), 1)
kontrol("Ciro (A: fct_satislar, B: dim_urunler)", fct["toplam_tutar"].sum(), urun["toplam_ciro"].sum(), 1)
kontrol("Kâr (A: fct_satislar, B: dim_urunler)", fct["kar"].sum(), urun["toplam_kar"].sum(), 1)
kontrol("Satılan adet (A: fct_satislar, B: dim_urunler)", fct["adet"].sum(), urun["toplam_satis_adedi"].sum())
kontrol("Sipariş sayısı (A: dim_musteriler toplamı, B: tekil sipariş)", musteri["siparis_sayisi"].sum(), fct["siparis_id"].nunique())
kontrol("Satır sayısı (A: fct_israf, B: mağaza+ürün+gün grup sayısı)", len(israf), fct.groupby(["magaza_id", "urun_adi", "siparis_tarihi"]).ngroups)
kontrol("Satılmayan adet = üretilen - satılan", israf["satilmayan_adet"].sum(), (israf["uretilen_adet"] - israf["satilan_adet"]).sum())
kontrol("Fire maliyeti = satılmayan adet x birim maliyet", israf["fire_maliyeti"].sum(), (israf["satilmayan_adet"] * israf["birim_maliyet"]).sum(), 1)
kontrol("RFM müşteri sayısı (A: dim_musteri_rfm, B: dim_musteriler)", len(rfm), len(musteri))

mutabakat = pd.DataFrame(sonuc)
mutabakat

## 4. Ana göstergeler (KPI)

In [ ]:
ciro = fct["toplam_tutar"].sum()
kar = fct["kar"].sum()
# Müşteri puanı sipariş düzeyinde verilir -> ortalamadan önce siparişleri tekilleştir
ort_puan = fct.drop_duplicates("siparis_id")["musteri_puani"].mean()

kpi = pd.Series({
    "Toplam ciro": ciro,
    "Toplam kâr": kar,
    "Kâr marjı (%)": kar / ciro * 100,
    "Toplam fire maliyeti": israf["fire_maliyeti"].sum(),
    "Fire maliyeti / ciro (%)": israf["fire_maliyeti"].sum() / ciro * 100,
    "Ortalama fire oranı (%)": israf["fire_orani"].mean(),
    "Ortalama müşteri puanı": ort_puan,
})
kpi.to_frame("değer")

## 5. Satış analizi

In [ ]:
aylik = fct.set_index("siparis_tarihi").resample("MS")[["toplam_tutar", "kar"]].sum()
ax = aylik.plot(figsize=(11, 4), marker="o", title="Aylık ciro ve kâr")
ax.set_ylabel("Tutar"); ax.set_xlabel("")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

saatlik = fct.groupby("saat")["adet"].sum()
saatlik.plot.bar(ax=axes[0], title="Saate göre toplam satış adedi")
axes[0].set_xlabel("Saat")

gunluk = (fct.groupby("siparis_tarihi")["adet"].sum().rename("adet").reset_index()
            .merge(tarih[["tarih", "haftanin_gunu_no", "haftanin_gunu"]], left_on="siparis_tarihi", right_on="tarih"))
ort_gun = gunluk.groupby(["haftanin_gunu_no", "haftanin_gunu"])["adet"].mean().reset_index()
sns.barplot(data=ort_gun, x="haftanin_gunu", y="adet", ax=axes[1], color="steelblue")
axes[1].set_title("Haftanın gününe göre ortalama günlük satış adedi")
axes[1].set_xlabel(""); axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout(); plt.show()

In [ ]:
# Hava durumu x ürün kategorisi: o hava durumundaki gün başına ortalama satış adedi
tmp = (fct.merge(urun[["urun_adi", "kategori"]], on="urun_adi")
          .merge(tarih[["tarih", "hava_durumu"]], left_on="siparis_tarihi", right_on="tarih"))

gun_sayisi = tarih["hava_durumu"].value_counts()
pivot = tmp.pivot_table(index="kategori", columns="hava_durumu", values="adet", aggfunc="sum")
pivot = pivot.div(gun_sayisi[pivot.columns], axis=1)

plt.figure(figsize=(9, 5))
sns.heatmap(pivot, annot=True, fmt=".0f", cmap="YlOrBr")
plt.title("Hava durumuna göre günlük ortalama satış adedi (kategori bazında)")
plt.ylabel(""); plt.xlabel("")
plt.show()

## 6. İsraf ve fire analizi

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

en_cok = israf.groupby("urun_adi")["fire_maliyeti"].sum().nlargest(10).sort_values()
en_cok.plot.barh(ax=axes[0], color="firebrick", title="Fire maliyeti en yüksek 10 ürün")
axes[0].set_ylabel("")

sira = [x for x in ["Low", "Medium", "High", "Very High"] if x in israf["bozulabilirlik"].unique()]
sns.barplot(data=israf, x="bozulabilirlik", y="fire_orani", order=sira, errorbar=None, ax=axes[1], color="darkorange")
axes[1].set_title("Bozulabilirlik seviyesine göre ortalama fire oranı (%)")
axes[1].set_xlabel("")

plt.tight_layout(); plt.show()

In [ ]:
magaza = pd.DataFrame({
    "ciro": fct.groupby("magaza_id")["toplam_tutar"].sum(),
    "fire_maliyeti": israf.groupby("magaza_id")["fire_maliyeti"].sum(),
})
magaza["fire / ciro (%)"] = magaza["fire_maliyeti"] / magaza["ciro"] * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
magaza[["ciro", "fire_maliyeti"]].plot.bar(ax=axes[0], title="Mağaza bazında ciro ve fire maliyeti", rot=0)
magaza["fire / ciro (%)"].plot.bar(ax=axes[1], title="Fire maliyetinin ciroya oranı (%)", rot=0, color="gray")
plt.tight_layout(); plt.show()
magaza

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
israf.groupby("hava_durumu")["fire_orani"].mean().sort_values().plot.barh(ax=axes[0], title="Hava durumuna göre ortalama fire oranı (%)")
israf.groupby("mevsim")["fire_orani"].mean().sort_values().plot.barh(ax=axes[1], title="Mevsime göre ortalama fire oranı (%)")
for a in axes: a.set_ylabel("")
plt.tight_layout(); plt.show()

In [ ]:
sayisal = ["uretilen_adet", "satilan_adet", "mevcut_stok", "satilmayan_adet",
           "fire_orani", "birim_maliyet", "birim_fiyat", "fire_maliyeti", "raf_omru_gun"]
plt.figure(figsize=(9, 7))
sns.heatmap(israf[sayisal].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("İsraf tablosunda değişkenler arası korelasyon")
plt.show()

## 7. Müşteri analizi (RFM)

In [ ]:
seg = (rfm.groupby("rfm_segmenti")
          .agg(musteri_sayisi=("musteri_id", "count"),
               ort_harcama=("toplam_harcama", "mean"),
               ort_kar=("toplam_kar", "mean"))
          .sort_values("musteri_sayisi"))

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
seg["musteri_sayisi"].plot.barh(ax=axes[0], title="Segmentlere göre müşteri sayısı")
seg[["ort_harcama", "ort_kar"]].plot.barh(ax=axes[1], title="Segment başına ortalama harcama ve kâr")
sns.boxplot(data=rfm, x="sadakat_uyesi_mi", y="rfm_ortalama_skor", ax=axes[2])
axes[2].set_title("Sadakat üyeliği ve RFM ortalama skoru")
for a in axes[:2]: a.set_ylabel("")
plt.tight_layout(); plt.show()
seg

## 8. Üretim tavsiyesi: mevcut üretim ile tavsiye edilen üretim

`fct_ideal_uretim_tavsiyesi` mağaza + ürün + haftanın günü için tavsiye üretir. Geçmişte gerçekte ortalama ne kadar üretildiğiyle karşılaştırıyoruz.
Bu karşılaştırma bir **göstergedir**, kesin tasarruf hesabı değildir.

In [ ]:
gercek = (israf.merge(tarih[["tarih", "haftanin_gunu_no"]], left_on="siparis_tarihi", right_on="tarih")
               .groupby(["magaza_id", "urun_adi", "haftanin_gunu_no"], as_index=False)
               .agg(ort_uretim=("uretilen_adet", "mean"),
                    ort_satis=("satilan_adet", "mean"),
                    ort_fire_maliyeti=("fire_maliyeti", "mean")))

karsi = gercek.merge(
    ideal[["magaza_id", "urun_adi", "haftanin_gunu_no", "tavsiye_edilen_uretim_adedi", "ortalama_gunluk_satis"]],
    on=["magaza_id", "urun_adi", "haftanin_gunu_no"], how="inner")
karsi["fark"] = karsi["ort_uretim"] - karsi["tavsiye_edilen_uretim_adedi"]

print(f"Eşleşen kombinasyon sayısı: {len(karsi)} / {len(ideal)}")
print(f"Mevcut üretimin tavsiyeden yüksek olduğu kombinasyonların oranı: %{(karsi['fark'] > 0).mean() * 100:.1f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(karsi["tavsiye_edilen_uretim_adedi"], karsi["ort_uretim"], alpha=0.6)
lim = [0, max(karsi["tavsiye_edilen_uretim_adedi"].max(), karsi["ort_uretim"].max())]
axes[0].plot(lim, lim, color="red", linestyle="--", label="eşitlik çizgisi")
axes[0].set_xlabel("Tavsiye edilen üretim"); axes[0].set_ylabel("Geçmiş ortalama üretim")
axes[0].set_title("Tavsiye vs mevcut üretim"); axes[0].legend()

sns.histplot(karsi["fark"], bins=30, ax=axes[1])
axes[1].set_title("Fark (mevcut ortalama üretim - tavsiye)")
plt.tight_layout(); plt.show()

## 9. Bulgular (kendi cümlelerinle doldur)

Grafikleri çalıştırdıktan sonra her madde için 1-2 cümle yaz:

- **Satış:** Ciro ve kâr zaman içinde nasıl değişiyor? En yoğun saat ve gün hangisi?
- **Hava durumu:** Hangi hava durumunda hangi kategori öne çıkıyor?
- **İsraf:** Fire maliyeti en yüksek ürünler hangileri? Bozulabilirlik arttıkça fire oranı artıyor mu?
- **Mağaza:** Hangi mağazada fire maliyeti / ciro oranı en yüksek?
- **Müşteri:** En büyük RFM segmenti hangisi? Sadakat üyeleri daha yüksek RFM skoruna sahip mi?
- **Üretim tavsiyesi:** Mevcut üretim çoğunlukla tavsiyenin üstünde mi altında mı? Bu ne anlama geliyor?
- **Mutabakat:** Hangi kontrollerde FARK çıktı ve nedeni neydi?